In [ ]:
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.exceptions import InvalidTag
from cryptography.hazmat.primitives.asymmetric import utils
import os

# Supported curves
CURVES = {
    "P-256": ec.SECP256R1,
    "P-384": ec.SECP384R1
}

# --- Key Generation (ECC) ---
def generate_key_pair(curve_name: str = "P-256"):
    if curve_name not in CURVES:
        raise ValueError(f"Unsupported curve {curve_name}")
    curve = CURVES[curve_name]()
    private_key = ec.generate_private_key(curve)
    public_key = private_key.public_key()

    # Serialize keys to hex
    private_bytes = private_key.private_bytes(
        encoding=serialization.Encoding.DER,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption()
    )
    public_bytes = public_key.public_bytes(
        encoding=serialization.Encoding.X962,
        format=serialization.PublicFormat.UncompressedPoint
    )
    return private_bytes.hex(), public_bytes.hex()

# --- Validate Public Key ---
def is_valid_public_key(hex_str: str, curve_name: str = "P-256") -> bool:
    try:
        curve = CURVES[curve_name]()
        ec.EllipticCurvePublicKey.from_encoded_point(
            curve,
            bytes.fromhex(hex_str)
        )
        return True
    except Exception:
        return False

# --- ECIES Encryption ---
def ecc_encrypt(public_key_hex: str, plaintext: str, curve_name: str = "P-256") -> str:
    if not is_valid_public_key(public_key_hex, curve_name):
        raise ValueError("Invalid public key format")

    curve = CURVES[curve_name]()
    public_key = ec.EllipticCurvePublicKey.from_encoded_point(
        curve,
        bytes.fromhex(public_key_hex)
    )

    # Generate ephemeral key pair
    ephemeral_private = ec.generate_private_key(curve)
    ephemeral_public = ephemeral_private.public_key()

    # ECDH key exchange
    shared_key = ephemeral_private.exchange(ec.ECDH(), public_key)

    # Derive AES-GCM key using HKDF
    derived_key = HKDF(
        algorithm=hashes.SHA256(),
        length=32,
        salt=None,
        info=b'ecies_encryption',
    ).derive(shared_key)

    # Encrypt message with AES-GCM
    aesgcm = AESGCM(derived_key)
    nonce = os.urandom(12)
    ciphertext = aesgcm.encrypt(nonce, plaintext.encode(), None)

    # Serialize ephemeral public key
    ephemeral_pub_bytes = ephemeral_public.public_bytes(
        encoding=serialization.Encoding.X962,
        format=serialization.PublicFormat.UncompressedPoint
    )

    # Return concatenated hex string: ephemeral_pub + nonce + ciphertext
    return ephemeral_pub_bytes.hex() + nonce.hex() + ciphertext.hex()

# --- ECIES Decryption ---
def ecc_decrypt(private_key_hex: str, encrypted_msg: str, curve_name: str = "P-256") -> str:
    curve = CURVES[curve_name]()
    private_key = serialization.load_der_private_key(
        bytes.fromhex(private_key_hex),
        password=None,
    )

    pub_key_len = (curve.key_size // 8) * 2 + 1  # Uncompressed point length in bytes
    ephemeral_pub_hex = encrypted_msg[:pub_key_len * 2]
    nonce_hex = encrypted_msg[pub_key_len * 2:pub_key_len * 2 + 24]
    ciphertext_hex = encrypted_msg[pub_key_len * 2 + 24:]

    ephemeral_public = ec.EllipticCurvePublicKey.from_encoded_point(
        curve,
        bytes.fromhex(ephemeral_pub_hex)
    )

    shared_key = private_key.exchange(ec.ECDH(), ephemeral_public)

    derived_key = HKDF(
        algorithm=hashes.SHA256(),
        length=32,
        salt=None,
        info=b'ecies_encryption',
    ).derive(shared_key)

    aesgcm = AESGCM(derived_key)
    try:
        plaintext = aesgcm.decrypt(
            bytes.fromhex(nonce_hex),
            bytes.fromhex(ciphertext_hex),
            None
        )
        return plaintext.decode()
    except InvalidTag:
        return "Error: authentication failed – ciphertext may be corrupted"
    except Exception as e:
        return f"Error: {str(e)}"

# --- Message Signing ---
def sign_message(private_key_hex: str, message: str) -> str:
    private_key = serialization.load_der_private_key(
        bytes.fromhex(private_key_hex),
        password=None
    )
    signature = private_key.sign(
        message.encode(),
        ec.ECDSA(hashes.SHA256())
    )
    return signature.hex()

# --- Signature Verification ---
def verify_signature(public_key_hex: str, message: str, signature_hex: str, curve_name: str = "P-256") -> bool:
    curve = CURVES[curve_name]()
    public_key = ec.EllipticCurvePublicKey.from_encoded_point(
        curve,
        bytes.fromhex(public_key_hex)
    )
    try:
        public_key.verify(
            bytes.fromhex(signature_hex),
            message.encode(),
            ec.ECDSA(hashes.SHA256())
        )
        return True
    except Exception:
        return False

# --- Key Save/Load (Optional) ---
def save_key(filename: str, key_hex: str):
    with open(filename, 'wb') as f:
        f.write(bytes.fromhex(key_hex))

def load_key(filename: str) -> str:
    with open(filename, 'rb') as f:
        return f.read().hex()

# --- Example Usage ---
if __name__ == "__main__":
    print("Generating ECC key pair...")
    priv_hex, pub_hex = generate_key_pair("P-256")
    print("Private Key (hex):", priv_hex)
    print("Public Key (hex):", pub_hex)

    message = "A HYBRID ELLIPTICAL CURVE CRYPTOGRAPHY TECHNIQUE FOR SECURED COMMUNICATION"
    print("\nOriginal Message:", message)

    print("\nEncrypting message...")
    encrypted = ecc_encrypt(pub_hex, message, "P-256")
    print("Encrypted Message (hex):", encrypted)

    print("\nDecrypting message...")
    decrypted = ecc_decrypt(priv_hex, encrypted, "P-256")
    print("Decrypted Message:", decrypted)

    print("\nSigning message...")
    signature = sign_message(priv_hex, message)
    print("Signature (hex):", signature)

    print("\nVerifying signature...")
    valid = verify_signature(pub_hex, message, signature, "P-256")
    print("Signature valid:", valid)

In [ ]:
import cv2
import numpy as np
import secrets
import hashlib
from PIL import Image

# --- Simulated ECC Key Exchange ---
def generate_shared_key():
    privKey = secrets.randbits(256)
    pubKey = privKey * 2  # Dummy operation
    shared_secret = privKey * pubKey
    shared_bytes = str(shared_secret).encode()
    aes_key = hashlib.sha256(shared_bytes).digest()
    return aes_key  # 32 bytes

# --- Simple AES-like XOR encryption ---
def xor_encrypt_decrypt(data_bytes, key):
    key_len = len(key)
    return bytes([b ^ key[i % key_len] for i, b in enumerate(data_bytes)])

# --- Entropy Calculation ---
def calculate_entropy(image):
    histogram = cv2.calcHist([image], [0], None, [256], [0, 256]).ravel()
    prob = histogram / histogram.sum() + 1e-10
    return -np.sum(prob * np.log2(prob))

# --- Correlation ---
def calculate_correlation(original, decrypted):
    return np.corrcoef(original.flatten(), decrypted.flatten())[0, 1]

# --- Load image from local system ---
# --- Load image from local system ---
image_path = input("Enter the path to your image file: ")

image = None
try:
    pil_image = Image.open(image_path).convert("L")
    image = np.array(pil_image)
    print(f"Loaded image: {image_path} with shape {image.shape}")
except Exception as e:
    image = None
    print(f"Error loading image: {e}")

# Fallback if image not uploaded properly
if image is None:
    image = np.random.randint(0, 256, (256, 256), dtype=np.uint8)
    print("No valid image uploaded. Generated a random image.")

# Generate simulated shared AES key
key = generate_shared_key()

# Encrypt image
image_bytes = image.tobytes()
encrypted_bytes = xor_encrypt_decrypt(image_bytes, key)
encrypted_array = np.frombuffer(encrypted_bytes, dtype=np.uint8).reshape(image.shape)

# Decrypt image
decrypted_bytes = xor_encrypt_decrypt(encrypted_bytes, key)
decrypted_image = np.frombuffer(decrypted_bytes, dtype=np.uint8).reshape(image.shape)

# Save outputs
cv2.imwrite("encrypted_image.png", encrypted_array)
cv2.imwrite("decrypted_image.png", decrypted_image)

# Calculate statistics
entropy_original = calculate_entropy(image)
entropy_encrypted = calculate_entropy(encrypted_array)
correlation = calculate_correlation(image, decrypted_image)

# Output results
print(f"Entropy of Original Image: {entropy_original:.4f}")
print(f"Entropy of Encrypted Image: {entropy_encrypted:.4f}")
print(f"Correlation between Original and Decrypted Image: {correlation:.4f}")

# Basic histogram display (numeric)
original_hist = cv2.calcHist([image], [0], None, [256], [0, 256])
encrypted_hist = cv2.calcHist([encrypted_array], [0], None, [256], [0, 256])

print("\nHistogram Stats:")
print(f"Original - Max Bin: {np.argmax(original_hist)}, Count: {np.max(original_hist)}")
print(f"Encrypted - Max Bin: {np.argmax(encrypted_hist)}, Count: {np.max(encrypted_hist)}")

In [ ]:
#!/usr/bin/env python3
"""
Comprehensive Elliptic Curve Cryptography (ECC) Analysis System — FIXED
========================================================================

This program provides detailed step-by-step analysis of ECC operations:
- Block 1: ECC Key Pair Generation (5 Steps)
- Block 2: Public Key Validation (4 Steps)
- Block 3: ECIES Encryption (8 Steps)
- Block 4: ECIES Decryption (8 Steps)
- Block 5: ECDSA Digital Signing (6 Steps)
- Block 6: ECDSA Signature Verification (7 Steps)

Fixes vs your original:
- ✅ Working @measure_performance decorator for instance methods
- ✅ Correct public key length check for curves like P-521 (uses ceil)
- ✅ Escaped braces in f-strings
- ✅ Cleaned up a few unicode/print glitches
- ✅ Minor robustness/clarity tweaks (no logic changes)

Author: ECC Analysis System (patched by ChatGPT)
Version: 2.1
"""

import os
import time
import hashlib
import matplotlib.pyplot as plt
import numpy as np
import math
from typing import Dict, List, Tuple, Any
import warnings
warnings.filterwarnings('ignore')

# Try to import cryptography library, install if not available
try:
    from cryptography.hazmat.primitives.asymmetric import ec
    from cryptography.hazmat.primitives import hashes, serialization
    from cryptography.hazmat.primitives.kdf.hkdf import HKDF
    from cryptography.hazmat.primitives.ciphers.aead import AESGCM
    from cryptography.exceptions import InvalidTag
except ImportError:
    print("Installing required cryptography library...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'cryptography', 'matplotlib'])
    from cryptography.hazmat.primitives.asymmetric import ec
    from cryptography.hazmat.primitives import hashes, serialization
    from cryptography.hazmat.primitives.kdf.hkdf import HKDF
    from cryptography.hazmat.primitives.ciphers.aead import AESGCM
    from cryptography.exceptions import InvalidTag

# --- Utility decorator (fixed): works with instance methods ---

def measure_performance(operation_name: str):
    """Decorator to time instance methods and store metrics on self."""
    def decorator(func):
        def wrapper(self, *args, **kwargs):
            start_time = time.perf_counter()
            result = func(self, *args, **kwargs)
            end_time = time.perf_counter()
            execution_time = (end_time - start_time) * 1000.0  # ms
            self.performance_data[operation_name] = execution_time
            self.step_timings.append({
                'operation': operation_name,
                'time_ms': execution_time,
                'timestamp': time.time()
            })
            print(f"   [⏱] {operation_name}: {execution_time:.4f} ms")
            return result
        return wrapper
    return decorator


class ComprehensiveECCAnalyzer:
    """
    Advanced ECC Analysis System with comprehensive security validation.
    Provides detailed step-by-step analysis of all cryptographic operations.
    """

    def __init__(self, curve_name: str = "P-256"):
        """Initialize analyzer with secure defaults."""
        self.curve_name = curve_name
        self.curves = {
            "P-256": ec.SECP256R1,
            "P-384": ec.SECP384R1,
            "P-521": ec.SECP521R1
        }
        if curve_name not in self.curves:
            raise ValueError(f"Unsupported curve: {curve_name}")

        self.curve = self.curves[curve_name]()
        self.performance_data: Dict[str, float] = {}
        self.security_metrics: Dict[str, Any] = {}
        self.step_timings: List[Dict[str, Any]] = []

        # Security configuration
        self.min_key_size = 256  # Minimum field size in bits
        self.required_entropy = 256  # Required entropy bits
        self.max_signature_age = 3600  # Max signature validity (seconds)

        print(f"   ECC Analyzer initialized with {curve_name}")
        print(f"   Security Level: ~{self.curve.key_size // 2} bits")
        print(f"   Field Size: {self.curve.key_size} bits")

    # --- Security helpers ---

    def validate_security_strength(self, data: bytes, min_entropy: int = 128) -> Dict[str, Any]:
        """Simple entropy estimate based on byte frequency (for demo/analysis)."""
        if len(data) == 0:
            return {'valid': False, 'reason': 'Empty data'}

        byte_counts = [0] * 256
        for b in data:
            byte_counts[b] += 1

        entropy_h = 0.0
        n = len(data)
        for count in byte_counts:
            if count:
                p = count / n
                entropy_h -= p * np.log2(p)
        estimated_entropy = entropy_h * n
        return {
            'valid': estimated_entropy >= min_entropy,
            'entropy_bits': estimated_entropy,
            'min_required': min_entropy,
            'strength_ratio': (estimated_entropy / min_entropy) if min_entropy else 0,
            'security_level': 'HIGH' if estimated_entropy >= min_entropy else 'LOW'
        }

    def print_security_analysis(self, data: bytes, name: str):
        s = self.validate_security_strength(data)
        print(f"  [ ] {name} Security Analysis:")
        print(f"     • Size: {len(data)} bytes ({len(data) * 8} bits)")
        print(f"     • Entropy: {s['entropy_bits']:.2f} bits")
        print(f"     • Security Level: {s['security_level']}")
        print(f"     • Strength Ratio: {s['strength_ratio']:.2f}x")
        print(f"     • SHA-256: {hashlib.sha256(data).hexdigest()[:16]}…")

    # --- BLOCK 1 ---

    @measure_performance("Block1_Step1_Curve_Init")
    def block1_step1_curve_initialization(self):
        print("\n" + '='*70)
        print(" BLOCK 1 - STEP 1: CURVE PARAMETER INITIALIZATION")
        print('='*70)

        print("Advanced Curve Analysis:")
        print(f"   • Curve Name: {self.curve_name}")
        print(f"   • Field Size: {self.curve.key_size} bits")
        print(f"   • Security Level: ~{self.curve.key_size // 2} bits")
        print("   • Curve Equation: y^2 = x^3 - 3x + b (mod p)")

        if self.curve.key_size < self.min_key_size:
            raise ValueError(f"Insufficient key size: {self.curve.key_size} < {self.min_key_size}")

        security_level = self.curve.key_size // 2
        self.security_metrics['curve_security'] = security_level

        print("   • Security Validation:PASSED")
        print("   • Quantum Resistance: Vulnerable to Shor's algorithm")
        print("   • Classical Resistance: No known efficient attacks")
        return {'status': 'success', 'security_level': security_level}

    @measure_performance("Block1_Step2_Private_Key_Gen")
    def block1_step2_private_key_generation(self):
        print("\n" + '='*70)
        print("BLOCK 1 - STEP 2: PRIVATE KEY GENERATION")
        print('='*70)

        print("Advanced Random Generation:")
        print("   • Source: Cryptographically secure PRNG (OS entropy)")
        print("   • Range: [1, n-1] where n is curve order")
        print("   • Security: High entropy")

        private_key = ec.generate_private_key(self.curve)
        private_bytes = private_key.private_bytes(
            encoding=serialization.Encoding.DER,
            format=serialization.PrivateFormat.PKCS8,
            encryption_algorithm=serialization.NoEncryption()
        )
        self.print_security_analysis(private_bytes, "Private Key")
        print("   • Storage Format: PKCS#8 DER encoding")
        return private_key, private_bytes

    @measure_performance("Block1_Step3_Public_Key_Derivation")
    def block1_step3_public_key_derivation(self, private_key):
        print("\n" + '='*70)
        print("BLOCK 1 - STEP 3: PUBLIC KEY DERIVATION")
        print('='*70)

        print("⚡ Elliptic Curve Point Multiplication:")
        print("   • Operation: Q = d × G (scalar multiplication)")
        print("   • Complexity: O(log n) via double-and-add")

        public_key = private_key.public_key()
        public_bytes = public_key.public_bytes(
            encoding=serialization.Encoding.X962,
            format=serialization.PublicFormat.UncompressedPoint
        )
        self.print_security_analysis(public_bytes, "Public Key")
        print("   • Point Format: X9.62 Uncompressed (0x04 | X | Y)")
        return public_key, public_bytes

    # --- BLOCK 2 ---

    @measure_performance("Block2_Public_Key_Validation")
    def block2_public_key_validation(self, public_bytes: bytes):
        print("\n" + '='*70)
        print("🔍 BLOCK 2: PUBLIC KEY VALIDATION (4 STEPS)")
        print('='*70)

        # Step 1: Format validation
        print("Step 1: Format Structure Validation")
        coord_len = math.ceil(self.curve.key_size / 8)
        expected_len = 1 + 2 * coord_len
        if len(public_bytes) != expected_len:
            raise ValueError(f"Invalid public key length: {len(public_bytes)} (expected {expected_len})")
        if public_bytes[0] != 0x04:
            raise ValueError("Invalid uncompressed point prefix (expected 0x04)")
        print("   • Format: X9.62 Uncompressed")

        # Step 2: Coordinate extraction
        print("Step 2: Coordinate Extraction")
        x_coord = public_bytes[1:1+coord_len]
        y_coord = public_bytes[1+coord_len:]
        print(f"   • X-coordinate: {len(x_coord)} bytes")
        print(f"   • Y-coordinate: {len(y_coord)} bytes")

        # Step 3: Point reconstruction
        print("Step 3: Elliptic Curve Point Reconstruction")
        try:
            _ = ec.EllipticCurvePublicKey.from_encoded_point(self.curve, public_bytes)
            print("   • Point reconstruction: SUCCESS")
        except Exception as e:
            print(f"   • Point reconstruction: FAILED - {e}")
            return False

        # Step 4: (Informational) Mathematical validation
        print("📄 Step 4: Mathematical Curve Equation Validation (library ensures this)")
        print("   • Equation check: y^2 ≡ x^3 - 3x + b (mod p) ")
        return True

    # --- BLOCK 3 ---

    @measure_performance("Block3_ECIES_Encryption")
    def block3_ecies_encryption(self, public_key, message: str):
        print("\n" + '='*70)
        print("BLOCK 3: ECIES ENCRYPTION (8 STEPS)")
        print('='*70)

        message_bytes = message.encode('utf-8')
        print("Step 1: Input Validation and Message Preparation")
        print(f"   • Message length: {len(message_bytes)} bytes (UTF-8)")

        print("Step 2: Ephemeral Key Pair Generation (PFS)")
        ephemeral_private = ec.generate_private_key(self.curve)
        ephemeral_public = ephemeral_private.public_key()
        ephemeral_bytes = ephemeral_public.public_bytes(
            encoding=serialization.Encoding.X962,
            format=serialization.PublicFormat.UncompressedPoint
        )
        print("   • Ephemeral: one-time use → Perfect Forward Secrecy")

        print("Step 3: ECDH Key Exchange")
        shared_secret = ephemeral_private.exchange(ec.ECDH(), public_key)
        self.print_security_analysis(shared_secret, "ECDH Shared Secret")

        print("Step 4: HKDF Key Derivation")
        derived_key = HKDF(
            algorithm=hashes.SHA256(),
            length=32,
            salt=None,
            info=b'ecies_encryption_v2',
        ).derive(shared_secret)
        self.print_security_analysis(derived_key, "Derived AES Key")

        print("Step 5: Cryptographic Nonce Generation")
        nonce = os.urandom(12)  # 96-bit nonce for AES-GCM
        self.print_security_analysis(nonce, "AES-GCM Nonce")

        print("Step 6: AES-256-GCM Authenticated Encryption")
        aesgcm = AESGCM(derived_key)
        ciphertext_with_tag = aesgcm.encrypt(nonce, message_bytes, None)
        print(f"   • Ciphertext size (incl. tag): {len(ciphertext_with_tag)} bytes")

        print("Step 7: Ciphertext Component Assembly")
        final_ciphertext = ephemeral_bytes + nonce + ciphertext_with_tag
        print(f"   • Total ciphertext: {len(final_ciphertext)} bytes")

        print("Step 8: Hexadecimal Encoding for Transmission")
        encrypted_hex = final_ciphertext.hex()
        print(f"   • Hex length: {len(encrypted_hex)} characters")
        return encrypted_hex, final_ciphertext

    # --- BLOCK 4 ---

    @measure_performance("Block4_ECIES_Decryption")
    def block4_ecies_decryption(self, private_key, encrypted_hex: str):
        print("\n" + '='*70)
        print("BLOCK 4: ECIES DECRYPTION (8 STEPS)")
        print('='*70)

        encrypted_bytes = bytes.fromhex(encrypted_hex)
        print("Step 1: Ciphertext Input Processing")
        print(f"   • Ciphertext size: {len(encrypted_bytes)} bytes")

        print("Step 2: Ciphertext Structure Parsing")
        coord_len = math.ceil(self.curve.key_size / 8)
        pub_key_len = 1 + 2 * coord_len
        nonce_len = 12
        ephemeral_pub_bytes = encrypted_bytes[:pub_key_len]
        nonce = encrypted_bytes[pub_key_len:pub_key_len + nonce_len]
        ciphertext_with_tag = encrypted_bytes[pub_key_len + nonce_len:]
        print(f"   • Ephemeral public key: {len(ephemeral_pub_bytes)} bytes")
        print(f"   • Nonce: {len(nonce)} bytes")
        print(f"   • Ciphertext+tag: {len(ciphertext_with_tag)} bytes")

        print("Step 3: Ephemeral Public Key Reconstruction")
        ephemeral_public = ec.EllipticCurvePublicKey.from_encoded_point(self.curve, ephemeral_pub_bytes)
        print("   • Key reconstruction: SUCCESS")

        print("Step 4: ECDH Key Exchange (Receiver Side)")
        shared_secret = private_key.exchange(ec.ECDH(), ephemeral_public)
        print("   • Shared secret regenerated: SUCCESS")

        print("Step 5: HKDF Key Re-derivation")
        derived_key = HKDF(
            algorithm=hashes.SHA256(),
            length=32,
            salt=None,
            info=b'ecies_encryption_v2',
        ).derive(shared_secret)
        print("   • AES key re-derived: SUCCESS")

        print(" Step 6: AES-GCM Authenticated Decryption")
        try:
            aesgcm = AESGCM(derived_key)
            plaintext_bytes = aesgcm.decrypt(nonce, ciphertext_with_tag, None)
            print("   • Authentication verification: PASSED")
        except InvalidTag:
            print("   • Authentication verification:  FAILED")
            return "ERROR: Message authentication failed"

        print("\nStep 7: Message Reconstruction")
        decrypted_message = plaintext_bytes.decode('utf-8')
        print(f"   • Message length: {len(decrypted_message)} characters")
        return decrypted_message

    # --- BLOCK 5 ---

    @measure_performance("Block5_ECDSA_Signing")
    def block5_ecdsa_signing(self, private_key, message: str):
        print("\n" + '='*70)
        print(" BLOCK 5: ECDSA DIGITAL SIGNING (6 STEPS)")
        print('='*70)

        message_bytes = message.encode('utf-8')
        print(" Step 1: Message Preparation and Encoding")
        print(f"   • Size: {len(message_bytes)} bytes")

        print(" Step 2: SHA-256 Hash Computation")
        message_hash = hashlib.sha256(message_bytes).digest()
        self.print_security_analysis(message_hash, "Message Hash")

        print(" Step 3: ECDSA Signature Generation")
        signature_bytes = private_key.sign(message_bytes, ec.ECDSA(hashes.SHA256()))
        self.print_security_analysis(signature_bytes, "ECDSA Signature")

        print(" Step 4: Signature Security Analysis")
        print("   • Non-repudiation, authentication, integrity ✔")

        print(" Step 5: Signature Format Analysis")
        signature_hex = signature_bytes.hex()
        print("   • Format: ASN.1 DER encoded")
        print(f"   • Size: {len(signature_bytes)} bytes (hex {len(signature_hex)} chars)")

        print(" Step 6: Security Properties")
        print(f"   • Algorithm: ECDSA over {self.curve_name} with SHA-256")
        print(f"   • Signature security: ~{self.curve.key_size // 2}-bit equivalent")
        return signature_hex, signature_bytes

    # --- BLOCK 6 ---

    @measure_performance("Block6_ECDSA_Verification")
    def block6_ecdsa_verification(self, public_key, message: str, signature_bytes: bytes):
        print("\n" + '='*70)
        print(" BLOCK 6: ECDSA SIGNATURE VERIFICATION (7 STEPS)")
        print('='*70)

        message_bytes = message.encode('utf-8')
        print(" Step 1: Input Data Assembly and Validation")
        print(f"   • Message size: {len(message_bytes)} bytes")
        print(f"   • Signature size: {len(signature_bytes)} bytes")

        print(" Step 2: Message Hash Re-computation (SHA-256)")
        _ = hashlib.sha256(message_bytes).digest()

        print(" Step 3: Signature Format Analysis")
        print("   • Format: ASN.1 DER encoded")
        print("   • Structure: SEQUENCE {{ r INTEGER, s INTEGER }}")

        print(" Step 4: ECDSA Mathematical Verification (library)")
        print("   • Computes u1 = H(m) * s^-1 mod n, u2 = r * s^-1 mod n, checks x(P) ≡ r")

        print(" Step 5: Verification Execution")
        try:
            public_key.verify(signature_bytes, message_bytes, ec.ECDSA(hashes.SHA256()))
            verification_result = True
            print("   • Mathematical verification: SUCCESS")
        except Exception as e:
            verification_result = False
            print(f"   • Mathematical verification:  FAILED ({e})")

        print(" Step 6: Security Implications Analysis")
        if verification_result:
            print("   • AUTHENTICATION: verified ✔ | INTEGRITY: verified ✔ | NON-REPUDIATION: supported ✔")
        else:
            print("   • WARNING: Signature verification failed — authenticity/integrity NOT guaranteed")

        print(" Step 7: Final Security Assessment")
        security_score = 100 if verification_result else 0
        print(f"   • Security Score: {security_score}/100")
        print(f"   • Recommendation: {'ACCEPT' if verification_result else 'REJECT'}")
        print(f"   • Trust Level: {'HIGH' if verification_result else 'NONE'}")
        return verification_result

    # --- Graphs ---

    def generate_performance_graphs(self):
        print("\n" + '='*70)
        print(" GENERATING PERFORMANCE AND SECURITY GRAPHS")
        print('='*70)

        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

        # Graph 1: Operation Performance Comparison
        operations = list(self.performance_data.keys())
        times = list(self.performance_data.values())
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD']
        bars1 = ax1.bar(range(len(operations)), times, color=colors[:len(operations)])
        ax1.set_xlabel('ECC Operations')
        ax1.set_ylabel('Time (milliseconds)')
        ax1.set_title('ECC Operations Performance Analysis', fontweight='bold')
        ax1.set_xticks(range(len(operations)))
        ax1.set_xticklabels([op.replace('_', '\n') for op in operations], rotation=45, ha='right')
        for bar, t in zip(bars1, times):
            h = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., h, f'{t:.3f}ms', ha='center', va='bottom', fontsize=9)

        # Graph 2: Security Strength Comparison
        algorithms = ['ECC P-256', 'ECC P-384', 'ECC P-521', 'RSA-2048', 'RSA-3072', 'RSA-4096']
        security_levels = [128, 192, 256, 112, 128, 152]
        key_sizes = [65, 97, 133, 256, 384, 512]  # Approx public key sizes in bytes
        bars2 = ax2.bar(algorithms, security_levels, color=['#FF9999', '#66B2FF', '#99FF99', '#FFB366', '#FF66FF', '#66FFFF'])
        ax2.set_xlabel('Cryptographic Algorithms')
        ax2.set_ylabel('Security Level (bits)')
        ax2.set_title('Security Level Comparison: ECC vs RSA', fontweight='bold')
        ax2.tick_params(axis='x', rotation=45)
        for bar, lvl in zip(bars2, security_levels):
            h = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., h, f'{lvl} bits', ha='center', va='bottom', fontsize=9)

        # Graph 3: Key Size Efficiency
        bars3 = ax3.bar(algorithms, key_sizes, color=['#FF9999', '#66B2FF', '#99FF99', '#FFB366', '#FF66FF', '#66FFFF'])
        ax3.set_xlabel('Cryptographic Algorithms')
        ax3.set_ylabel('Public Key Size (bytes)')
        ax3.set_title('Key Size Efficiency: ECC vs RSA', fontweight='bold')
        ax3.tick_params(axis='x', rotation=45)
        for bar, sz in zip(bars3, key_sizes):
            h = bar.get_height()
            ax3.text(bar.get_x() + bar.get_width()/2., h, f'{sz}B', ha='center', va='bottom', fontsize=9)

        # Graph 4: Block Processing Time Distribution
        if self.step_timings:
            block_times: Dict[str, float] = {}
            for t in self.step_timings:
                block = t['operation'].split('_')[0]
                block_times[block] = block_times.get(block, 0.0) + t['time_ms']
            blocks = list(block_times.keys())
            tvals = list(block_times.values())
            ax4.pie(tvals, labels=blocks, autopct='%1.1f%%', colors=colors[:len(blocks)], startangle=90)
            ax4.set_title('Processing Time Distribution by Block', fontweight='bold')
        else:
            ax4.text(0.5, 0.5, 'No timings yet', ha='center')
            ax4.axis('off')

        plt.tight_layout()
        plt.savefig('ecc_comprehensive_analysis.png', dpi=300, bbox_inches='tight')
        plt.show()

        total_time = sum(self.performance_data.values()) if self.performance_data else 0.0
        if self.performance_data:
            fastest = min(self.performance_data.items(), key=lambda x: x[1])
            slowest = max(self.performance_data.items(), key=lambda x: x[1])
        else:
            fastest = slowest = ("N/A", 0.0)
        print("\n🏁 PERFORMANCE SUMMARY:")
        print(f"   • Total processing time: {total_time:.4f} ms")
        if self.performance_data:
            print(f"   • Average operation time: {total_time/len(self.performance_data):.4f} ms")
        print(f"   • Fastest operation: {fastest}")
        print(f"   • Slowest operation: {slowest}")


def main():
    """Main function to run comprehensive ECC analysis."""
    print(" COMPREHENSIVE ELLIPTIC CURVE CRYPTOGRAPHY ANALYSIS SYSTEM")
    print("=" * 80)
    print("This system provides detailed analysis of all ECC operations:")
    print("• Block 1: ECC Key Pair Generation (5 Steps)")
    print("• Block 2: Public Key Validation (4 Steps)")
    print("• Block 3: ECIES Encryption (8 Steps)")
    print("• Block 4: ECIES Decryption (8 Steps)")
    print("• Block 5: ECDSA Digital Signing (6 Steps)")
    print("• Block 6: ECDSA Signature Verification (7 Steps)")
    print("=" * 80)

    try:
        analyzer = ComprehensiveECCAnalyzer("P-256")

        test_message = "ADVANCED HYBRID ELLIPTIC CURVE CRYPTOGRAPHY FOR MAXIMUM SECURITY"
        print(f"\n Test Message: '{test_message}'")
        print(f" Message Analysis: {len(test_message)} characters, {len(test_message.encode())} bytes")

        # BLOCK 1: Key Generation Analysis
        print("\n Starting Block 1 Analysis…")
        analyzer.block1_step1_curve_initialization()
        private_key, private_bytes = analyzer.block1_step2_private_key_generation()
        public_key, public_bytes = analyzer.block1_step3_public_key_derivation(private_key)

        # BLOCK 2: Public Key Validation
        print("\n Starting Block 2 Analysis…")
        validation_result = analyzer.block2_public_key_validation(public_bytes)

        # BLOCK 3: ECIES Encryption
        print("\n Starting Block 3 Analysis…")
        encrypted_hex, encrypted_bytes = analyzer.block3_ecies_encryption(public_key, test_message)

        # BLOCK 4: ECIES Decryption
        print("\n Starting Block 4 Analysis…")
        decrypted_message = analyzer.block4_ecies_decryption(private_key, encrypted_hex)

        # BLOCK 5: ECDSA Signing
        print("\n Starting Block 5 Analysis…")
        signature_hex, signature_bytes = analyzer.block5_ecdsa_signing(private_key, test_message)

        # BLOCK 6: ECDSA Verification (Valid)
        print("\n Starting Block 6 Analysis (Valid Signature)…")
        verification_result = analyzer.block6_ecdsa_verification(public_key, test_message, signature_bytes)

        # BLOCK 6: ECDSA Verification (Invalid - Tampered Message)
        print("\n Starting Block 6 Analysis (Tampered Message)…")
        tampered_message = "TAMPERED MESSAGE - SHOULD FAIL VERIFICATION"
        tampered_result = analyzer.block6_ecdsa_verification(public_key, tampered_message, signature_bytes)

        # Generate comprehensive graphs
        analyzer.generate_performance_graphs()

        # Final Results Summary
        print("\n" + '='*80)
        print("🎯 COMPREHENSIVE ANALYSIS RESULTS")
        print('='*80)
        print(f" Original Message: '{test_message}'")
        print(f" Decrypted Message: '{decrypted_message}'")
        print(f" Encryption/Decryption: {'SUCCESS' if test_message == decrypted_message else 'FAILED'}")
        print(f" Valid Signature Verification: {'PASSED' if verification_result else 'FAILED'}")
        print(f" Tampered Signature Verification: {'PROPERLY REJECTED' if not tampered_result else 'SECURITY BREACH'}")

        print(f"\n  SECURITY ASSESSMENT:")
        print("   • Overall Security Level: HIGH")
        print("   • All cryptographic operations: SUCCESSFUL")
        print("   • Authentication mechanisms: WORKING")
        print("   • Integrity protection: ACTIVE")
        print("   • Perfect Forward Secrecy: ENABLED")
        print("   • System Status: SECURE ✅")

        print(f"\n SYSTEM PERFORMANCE:")
        total_time = sum(analyzer.performance_data.values()) if analyzer.performance_data else 0.0
        print(f"   • Total processing time: {total_time:.4f} ms")
        print(f"   • Operations completed: {len(analyzer.performance_data)}")
        if analyzer.performance_data:
            print(f"   • Average operation time: {total_time/len(analyzer.performance_data):.4f} ms")
        print("   • Performance rating: EXCELLENT")

        print("\n ANALYSIS COMPLETE - ALL SYSTEMS SECURE!")
        print("=" * 80)

    except Exception as e:
        print(f"\n CRITICAL ERROR: {e}")
        print("System analysis terminated due to security failure.")
        raise


if __name__ == "__main__":
    main()

In [ ]:
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import os

# Supported curves
CURVES = {
    "P-256": ec.SECP256R1,
    "P-384": ec.SECP384R1
}

# --- Key Generation (ECC) ---
def generate_key_pair(curve_name: str = "P-256"):
    if curve_name not in CURVES:
        raise ValueError(f"Unsupported curve {curve_name}")
    curve = CURVES[curve_name]()
    private_key = ec.generate_private_key(curve)
    public_key = private_key.public_key()

    # Serialize keys to hex
    private_bytes = private_key.private_bytes(
        encoding=serialization.Encoding.DER,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption()
    )
    public_bytes = public_key.public_bytes(
        encoding=serialization.Encoding.X962,
        format=serialization.PublicFormat.UncompressedPoint
    )
    return private_bytes.hex(), public_bytes.hex()

# --- Validate Public Key ---
def is_valid_public_key(hex_str: str, curve_name: str = "P-256") -> bool:
    try:
        curve = CURVES[curve_name]()
        ec.EllipticCurvePublicKey.from_encoded_point(
            curve,
            bytes.fromhex(hex_str)
        )
        return True
    except Exception:
        return False

# --- ECIES Encryption ---
def ecc_encrypt(public_key_hex: str, plaintext: str, curve_name: str = "P-256") -> str:
    if not is_valid_public_key(public_key_hex, curve_name):
        raise ValueError("Invalid public key format")

    curve = CURVES[curve_name]()
    public_key = ec.EllipticCurvePublicKey.from_encoded_point(
        curve,
        bytes.fromhex(public_key_hex)
    )

    # Generate ephemeral key pair
    ephemeral_private = ec.generate_private_key(curve)
    ephemeral_public = ephemeral_private.public_key()

    # ECDH key exchange
    shared_key = ephemeral_private.exchange(ec.ECDH(), public_key)

    # Derive AES-GCM key using HKDF
    derived_key = HKDF(
        algorithm=hashes.SHA256(),
        length=32,
        salt=None,
        info=b'ecies_encryption',
    ).derive(shared_key)

    # Encrypt message with AES-GCM
    aesgcm = AESGCM(derived_key)
    nonce = os.urandom(12)
    ciphertext = aesgcm.encrypt(nonce, plaintext.encode(), None)

    # Serialize ephemeral public key
    ephemeral_pub_bytes = ephemeral_public.public_bytes(
        encoding=serialization.Encoding.X962,
        format=serialization.PublicFormat.UncompressedPoint
    )

    # Return concatenated hex string: ephemeral_pub + nonce + ciphertext
    return ephemeral_pub_bytes.hex() + nonce.hex() + ciphertext.hex()

# --- Example Usage ---
if __name__ == "__main__":
    print("=== ECC ENCRYPTION MODULE ===")
    print("Generating ECC key pair...")
    priv_hex, pub_hex = generate_key_pair("P-256")
    print("Private Key (hex):", priv_hex)
    print("Public Key (hex):", pub_hex)

    message = "A HYBRID ELLIPTICAL CURVE CRYPTOGRAPHY TECHNIQUE FOR SECURED COMMUNICATION"
    print("\nOriginal Message:", message)

    print("\nEncrypting message...")
    encrypted = ecc_encrypt(pub_hex, message, "P-256")
    print("\n" + "="*60)
    print("ENCRYPTED MESSAGE (Copy this for decryption):")
    print("="*60)
    print(encrypted)
    print("\n" + "="*60)
    print("PRIVATE KEY (Copy this for decryption):")
    print("="*60)
    print(priv_hex)
    print("\nâœ… Encryption completed successfully!")
    print("\nNext: Copy both values above and paste them into the decryption module.")

In [ ]:
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.exceptions import InvalidTag

# Supported curves
CURVES = {
    "P-256": ec.SECP256R1,
    "P-384": ec.SECP384R1
}

# --- ECIES Decryption ---
def ecc_decrypt(private_key_hex: str, encrypted_msg: str, curve_name: str = "P-256") -> str:
    curve = CURVES[curve_name]()
    private_key = serialization.load_der_private_key(
        bytes.fromhex(private_key_hex),
        password=None,
    )

    pub_key_len = (curve.key_size // 8) * 2 + 1  # Uncompressed point length in bytes
    ephemeral_pub_hex = encrypted_msg[:pub_key_len * 2]
    nonce_hex = encrypted_msg[pub_key_len * 2:pub_key_len * 2 + 24]
    ciphertext_hex = encrypted_msg[pub_key_len * 2 + 24:]

    ephemeral_public = ec.EllipticCurvePublicKey.from_encoded_point(
        curve,
        bytes.fromhex(ephemeral_pub_hex)
    )

    shared_key = private_key.exchange(ec.ECDH(), ephemeral_public)

    derived_key = HKDF(
        algorithm=hashes.SHA256(),
        length=32,
        salt=None,
        info=b'ecies_encryption',
    ).derive(shared_key)

    aesgcm = AESGCM(derived_key)
    try:
        plaintext = aesgcm.decrypt(
            bytes.fromhex(nonce_hex),
            bytes.fromhex(ciphertext_hex),
            None
        )
        return plaintext.decode()
    except InvalidTag:
        return "Error: authentication failed â€“ ciphertext may be corrupted"
    except Exception as e:
        return f"Error: {str(e)}"

# --- Example Usage ---
if __name__ == "__main__":
    print("=== ECC DECRYPTION MODULE ===")
    print("Please provide the following inputs from the encryption module:")

    print("\n" + "="*60)
    # Input encrypted message
    encrypted_message = input("Enter the encrypted message (hex): ").strip()

    print("\n" + "="*60)
    # Input private key
    private_key = input("Enter the private key (hex): ").strip()

    print("\n" + "="*60)
    print("DECRYPTION IN PROGRESS...")
    print("="*60)

    # Decrypt the message
    try:
        decrypted_message = ecc_decrypt(private_key, encrypted_message, "P-256")

        print("\n" + "="*60)
        print("DECRYPTION RESULT:")
        print("="*60)
        print("Original Message:", decrypted_message)
        print("\nâœ… Decryption completed successfully!")

    except Exception as e:
        print(f"\nâŒ Decryption failed!")
        print(f"Error: {e}")
        print("\nPossible causes:")
        print("1. Incorrect encrypted message format")
        print("2. Wrong private key")
        print("3. Data corruption during copy/paste")
        print("4. Mismatched curve parameters")